# 07 · Describir el entorno antes de decidir

Construimos diez características: números que describen la tendencia, el riesgo, el crédito y la situación de las industrias. Más adelante el modelo los utilizará para decidir si acepta una señal. Aquí **no calculamos etiquetas ni entrenamos modelos**, tampoco elegimos variables por su rentabilidad.

Necesitas haber ejecutado el **notebook 01**. El 02 y las simulaciones no son necesarios. Reutilizamos los originales de industrias y crédito del diagnóstico 00, si faltan, se descargan esas tres fuentes oficiales. No se descarga Yahoo ni se utilizan sus cierres en esta tabla. No hay archivos de configuración.

Se conserva el desarrollo 1927–1989. Los promedios de Shiller siguen siendo una aproximación histórica, no precios de ejecución. El dividendo D y RF no entran como características.


## 1. Leer las fuentes

`target_period` es el mes cuyo resultado queremos evaluar, `decision_period` es el mes anterior. Los precios llegan hasta la decisión. Para crédito e industrias utilizamos un mes adicional de retraso: si el objetivo es mayo de 1975, usamos precios hasta abril y contexto hasta marzo.

Este retraso es una convención explícita. No reconstruye las versiones históricas de las fuentes ni demuestra cuándo se publicó cada dato. Las medias mensuales de Shiller tampoco se convierten en cotizaciones disponibles para ejecutar una orden al cierre.


In [ ]:
import pandas as pd
from IPython.display import display

from tfm.data import project_root, load_monthly, save_result
from tfm.diagnostics import load_context_sources, SOURCE_URLS
from tfm.features import build_monthly_features, feature_dictionary, FEATURE_COLUMNS

ROOT = project_root()
mensuales = load_monthly(ROOT)
contexto = load_context_sources(ROOT)
print("Desarrollo: enero de 1927 → diciembre de 1989. No se utiliza la evaluación desde 1990.")
display(pd.DataFrame([
    {"Fuente": nombre, "Primer mes": tabla.period.iloc[0], "Último mes": tabla.period.iloc[-1],
     "Observaciones": len(tabla), "URL": SOURCE_URLS[nombre]}
    for nombre, tabla in {"shiller": mensuales, **contexto}.items()
]))


## 2. Qué significa cada característica

En las fórmulas, **m** es el mes de decisión y **k = m−1** el último mes de contexto utilizado. Los intervalos incluyen ambos extremos. Una distancia de 0,05 significa estar un 5 % por encima de la media, un drawdown de −0,10 significa estar un 10 % por debajo del máximo de los últimos doce meses. No es la mayor caída sufrida dentro de toda esa ventana.

La volatilidad usa doce variaciones mensuales y desviación típica muestral, multiplicada por la raíz de doce. El momentum reutiliza exactamente el indicador de la regla: excluye el mes más reciente y comprende once variaciones. La distancia a la media sí incluye el precio del mes de decisión.

El retorno acumulado de **24 meses** compara el precio del mes de decisión con el de dos años antes: `P[m] / P[m−24] − 1`. Sustituye a la distancia a la media de tres meses para describir un horizonte más largo y conectar con el comparador bajista del notebook 05. Aquí conservamos la magnitud de la subida o caída; el filtro sencillo solo utiliza si el valor es negativo. No añadimos otra variable binaria.

La sustitución se decide por el fundamento del estudio, antes de definir etiquetas o entrenar modelos; no se ha demostrado que prediga mejor ni que deje de estar relacionada con momentum. La dispersión industrial sigue siendo exploratoria. La referencia de dos años procede de [Daniel y Moskowitz (2016), sección 3.2](https://kentdaniel.net/papers/published/jfe_16.pdf), adaptada a nuestros precios medios sin dividendos. La historia desde diciembre de 1924 ya permite calcular el primer objetivo de enero de 1927.

El diferencial Baa–Aaa compara dos rendimientos corporativos de distinta calidad crediticia, no es un retorno de una inversión ni un diferencial frente a deuda pública. Si Baa es 8 % y Aaa es 6 %, el diferencial es **2 puntos porcentuales**, no 2 % de rentabilidad mensual. Su variación es la diferencia respecto al mes anterior, también en puntos porcentuales.

Para industrias reconstruimos índices de riqueza `W` multiplicando los factores `(1 + retorno)` de las doce carteras de French. La amplitud cuenta la fracción de índices por encima de su propia media de doce observaciones, una igualdad no cuenta como superación. No se comparan los retornos directamente con una media de precios. La dispersión es la desviación típica muestral de los doce retornos del mes. Son carteras industriales ponderadas por valor, no doce acciones ni ETF, y su universo no coincide con los componentes del S&P 500.


In [ ]:
diccionario = feature_dictionary()
display(diccionario)


## 3. Construir la tabla y comprobar el calentamiento

Conservamos los 756 meses y las ausencias iniciales. `features_ready` indica si están completas las diez variables, es una marca de disponibilidad, no una etiqueta que el modelo deba aprender. Las columnas de fechas y activo tampoco son predictores.

No rellenamos huecos, no estandarizamos con todo el periodo ni seleccionamos variables. Esos ajustes, cuando correspondan, se aprenderán dentro de cada partición de entrenamiento. La correlación entre características tampoco justifica eliminarlas aquí: se estudiará más adelante con el protocolo de selección y Clustered MDA.


In [ ]:
caracteristicas = build_monthly_features(
    mensuales, contexto["industries"], contexto["BAA"], contexto["AAA"]
)
utilizables = caracteristicas.loc[caracteristicas.features_ready].copy()
cobertura = pd.DataFrame([
    {"Variable": columna, "Meses disponibles": int(caracteristicas[columna].notna().sum()),
     "Primer objetivo disponible": caracteristicas.loc[caracteristicas[columna].notna(), "target_period"].iloc[0]}
    for columna in FEATURE_COLUMNS
])
display(cobertura)
print(f"Meses conservados: {len(caracteristicas)} | Con diez variables completas: {len(utilizables)}")
print(f"Muestra completa: {utilizables.target_period.iloc[0]} → {utilizables.target_period.iloc[-1]}")
display(caracteristicas.head(9))


## 4. Entender un mes concreto

Cambia solo `mes_objetivo` si quieres inspeccionar otra fila del desarrollo. Las fechas impresas permiten distinguir el mes que describimos del mes cuyo resultado evaluaremos. La tabla de precios muestra los puntos que intervienen en las dos reglas y el retorno de 24 meses, no contiene la rentabilidad posterior.


In [ ]:
mes_objetivo = "1975-05"
if mes_objetivo not in set(caracteristicas.target_period):
    raise ValueError("Escoge un mes objetivo entre 1927-01 y 1989-12")
fila = caracteristicas.set_index("target_period").loc[mes_objetivo]
decision = pd.Period(fila.decision_period, freq="M")
mes_contexto = pd.Period(fila.context_source_period, freq="M")
print(f"Objetivo: {mes_objetivo} | Decisión/precios hasta: {decision} | Contexto hasta: {mes_contexto}")
precios = mensuales.set_index("period").price_monthly_mean
display(pd.DataFrame([
    {"Uso": "Retorno 24 meses: inicio", "Mes": str(decision - 24), "P": precios.loc[str(decision - 24)]},
    {"Uso": "Momentum: inicio", "Mes": str(decision - 12), "P": precios.loc[str(decision - 12)]},
    {"Uso": "Momentum: fin / reversión: inicio", "Mes": str(decision - 1), "P": precios.loc[str(decision - 1)]},
    {"Uso": "Reversión / retorno 24 meses: fin", "Mes": str(decision), "P": precios.loc[str(decision)]},
]))
display(fila[FEATURE_COLUMNS].rename("Valor de la característica").to_frame())


## 5. Ver de qué fecha llega el contexto

La variación de crédito necesita dos meses de fuente, ambos anteriores a la decisión. La amplitud necesita doce observaciones de riqueza, la primera se obtiene al terminar julio de 1926. La base inicial de riqueza es arbitraria y no altera estar por encima o por debajo de su propia media.

Los primeros siete objetivos de 1927 carecen de amplitud con este retraso. Para comparaciones que incluyan las diez variables utilizaremos la misma muestra completa, sin reemplazar esos valores por cero. Esta reducción corresponde al calentamiento, no a ampliar ni trasladar el periodo de entrenamiento de la guía.


In [ ]:
credito = pd.DataFrame({
    "Baa (%)": contexto["BAA"].set_index("period").yield_percent,
    "Aaa (%)": contexto["AAA"].set_index("period").yield_percent,
})
credito["Diferencial (p.p.)"] = credito["Baa (%)"] - credito["Aaa (%)"]
display(credito.loc[[str(mes_contexto - 1), str(mes_contexto)]])
retornos_industriales = contexto["industries"].set_index("period").loc[str(mes_contexto)]
display((100 * retornos_industriales).rename("Retorno industrial del mes de fuente (%)").to_frame())
print("La dispersión usa estos doce retornos, la amplitud utiliza sus índices de riqueza y doce meses de historia.")
print("Las columnas en decimal se multiplican por 100 solo al mostrarlas como porcentaje, crédito ya está en p.p.")


## 6. Guardar la tabla explicada

**Esta celda guarda o sustituye** `data/processed/monthly_features.parquet` y su resumen JSON. Conserva las 756 filas, incluyendo las siete ausencias iniciales de amplitud. No cambia originales, señales, componentes ni simulaciones. El JSON describe las fórmulas y los supuestos utilizados, no es un archivo de configuración.

Si vuelves a preparar Shiller en el notebook 01, se retira esta tabla y debes ejecutar de nuevo el 07. Si actualizas los originales de contexto, también debes volver a ejecutar el 07. La función no recarga ni sustituye originales que ya existan.


In [ ]:
informe = {
    "first_target": caracteristicas.target_period.iloc[0],
    "last_target": caracteristicas.target_period.iloc[-1],
    "complete_rows": len(utilizables),
    "first_complete_target": utilizables.target_period.iloc[0],
    "last_complete_target": utilizables.target_period.iloc[-1],
    "incomplete_targets": caracteristicas.loc[~caracteristicas.features_ready, "target_period"].tolist(),
    "feature_columns": FEATURE_COLUMNS,
    "dictionary": diccionario.to_dict("records"),
    "feature_design": "Retorno de 24 meses sustituye a distancia a media de 3 meses por fundamento conceptual, antes de entrenar; sin mejora predictiva demostrada.",
    "timing": {"price_source": "target - 1 month", "context_source": "target - 2 months"},
    "warnings": [
        "Precios medios de Shiller, no cierres negociables. No se utilizan D ni RF como predictores.",
        "El retraso de contexto es un supuesto, no acredita versiones ni fechas históricas de publicación.",
        "Industrias de French, no dispersión entre acciones individuales ni amplitud de ETF del S&P 500.",
        "Ausencias de calentamiento sin rellenar. No se han seleccionado características por resultados.",
        "Sin etiquetas, modelos, estandarización global ni evaluación desde 1990.",
    ],
}
resumen = save_result(ROOT, "monthly_features", caracteristicas, informe)
print(f"Guardado: {resumen['path']}")
print(f"{resumen['rows']} filas, {resumen['complete_rows']} con las diez características completas.")


## Qué hemos preparado y qué falta

Esta tabla describe el entorno anterior a cada mes objetivo. Las mismas características podrán acompañar a las señales de momentum y reversión, el sentido de cada operación sigue viniendo de la regla original. El modelo todavía no sabe qué señales habrían sido favorables: ese será el siguiente incremento, al definir las etiquetas y su relación con los costes.

Fuentes y alcance: [Shiller](https://shillerdata.com/), [doce industrias de French](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_12_ind_port.html), [BAA](https://fred.stlouisfed.org/series/BAA) y [AAA](https://fred.stlouisfed.org/series/AAA) de Moody's obtenidas en FRED. El análisis de cobertura y de medias frente a cierres se conserva en `docs/diagnostico_fuentes.md`.
